# 01 - Exploración de Datos: CSE-CIC-IDS2018

Exploración del dataset CSE-CIC-IDS2018 para el proyecto Neural ODE Autoencoder.

**Objetivos:**
1. Inventario de archivos y estructura
2. Verificar consistencia de schema entre archivos
3. Identificar y cuantificar problemas conocidos (headers duplicados, NaN, Inf)
4. Distribución de labels (benigno vs. ataques)
5. Estadísticas descriptivas de features clave
6. Distribución temporal de flujos
7. Visualizaciones comparativas benigno vs. DDoS

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

RAW_DIR = os.path.join("..", "data", "raw")
print(f"Buscando CSVs en: {os.path.abspath(RAW_DIR)}")

## 1. Inventario de archivos

In [ ]:
csv_files = sorted([f for f in os.listdir(RAW_DIR) if f.endswith(".csv")])

print(f"Total de archivos CSV: {len(csv_files)}\n")
for f in csv_files:
    size_mb = os.path.getsize(os.path.join(RAW_DIR, f)) / 1e6
    print(f"  {f:60s} {size_mb:8.1f} MB")

## 2. Carga y verificación de schema

Cargamos cada archivo individualmente para verificar consistencia de columnas.

In [ ]:
# Leer las primeras filas de cada archivo para verificar columnas
column_sets = {}
sample_dfs = {}

for f in csv_files:
    path = os.path.join(RAW_DIR, f)
    df_sample = pd.read_csv(path, nrows=5, encoding="utf-8", low_memory=False)
    df_sample.columns = df_sample.columns.str.strip()
    column_sets[f] = set(df_sample.columns)
    sample_dfs[f] = df_sample
    print(f"\n--- {f} ---")
    print(f"  Columnas: {len(df_sample.columns)}")
    print(f"  Primeras columnas: {list(df_sample.columns[:5])}")

# Verificar si todos los archivos tienen las mismas columnas
ref_cols = column_sets[csv_files[0]]
all_same = all(cols == ref_cols for cols in column_sets.values())
print(f"\n{'✓' if all_same else '✗'} Todas las columnas son consistentes: {all_same}")

if not all_same:
    for f, cols in column_sets.items():
        diff = cols.symmetric_difference(ref_cols)
        if diff:
            print(f"  {f}: diferencias = {diff}")

## 3. Carga completa y limpieza inicial

Cargamos todos los CSVs, aplicamos limpieza básica para poder explorar.

In [ ]:
dfs = []
for f in csv_files:
    path = os.path.join(RAW_DIR, f)
    df = pd.read_csv(path, encoding="utf-8", low_memory=False)
    df.columns = df.columns.str.strip()
    df["_source_file"] = f
    dfs.append(df)
    print(f"  {f}: {len(df):>10,} filas")

df_all = pd.concat(dfs, ignore_index=True)
print(f"\nTotal de filas: {len(df_all):,}")
print(f"Total de columnas: {len(df_all.columns)}")

## 4. Problemas conocidos del dataset

Según la documentación (Sección 7.5):
- Headers duplicados embebidos como filas (Feb 16, Feb 28, Mar 1)
- Infinity y NaN en `Flow Bytes/s` y `Flow Packets/s`
- Columna duplicada `Fwd Header Length`

In [ ]:
# 4a. Detectar filas con headers duplicados
# Estas filas tienen el nombre de la columna como valor (ej: "Label" en la columna Label)
if "Label" in df_all.columns:
    header_rows = df_all[df_all["Label"] == "Label"]
    print(f"Filas con headers duplicados: {len(header_rows):,}")
    if len(header_rows) > 0:
        print("  En archivos:")
        print(header_rows["_source_file"].value_counts().to_string(header=False))
    # Eliminar headers duplicados
    df_all = df_all[df_all["Label"] != "Label"].reset_index(drop=True)
    print(f"\nFilas después de eliminar headers duplicados: {len(df_all):,}")

In [ ]:
# 4b. Infinity y NaN en Flow Bytes/s y Flow Packets/s
for col in ["Flow Bytes/s", "Flow Packets/s"]:
    if col in df_all.columns:
        # Convertir a numérico (Infinity viene como string a veces)
        df_all[col] = pd.to_numeric(df_all[col], errors="coerce")
        n_inf = np.isinf(df_all[col]).sum()
        n_nan = df_all[col].isna().sum()
        print(f"{col}:")
        print(f"  Infinity: {n_inf:,}")
        print(f"  NaN:      {n_nan:,}")
        # Reemplazar Inf con NaN para imputación posterior
        df_all[col] = df_all[col].replace([np.inf, -np.inf], np.nan)

In [ ]:
# 4c. Columna duplicada Fwd Header Length
fwd_header_cols = [c for c in df_all.columns if "Fwd Header Length" in c]
print(f"Columnas con 'Fwd Header Length': {fwd_header_cols}")
if len(fwd_header_cols) > 1:
    print(f"  → Columna duplicada detectada. Se eliminará una de las copias en el preprocesamiento.")

## 5. Distribución de labels

In [ ]:
# Distribución general de labels
label_counts = df_all["Label"].value_counts()
label_pct = (label_counts / len(df_all) * 100).round(2)

label_summary = pd.DataFrame({"Count": label_counts, "Percentage": label_pct})
print(label_summary.to_string())

print(f"\nTotal benigno: {label_counts.get('Benign', 0):,} ({label_pct.get('Benign', 0):.1f}%)")
print(f"Total ataques: {(len(df_all) - label_counts.get('Benign', 0)):,} ({100 - label_pct.get('Benign', 0):.1f}%)")

In [ ]:
# Distribución de labels por archivo
fig, ax = plt.subplots(figsize=(14, 6))

attack_labels = [l for l in df_all["Label"].unique() if l != "Benign"]
pivot = df_all.groupby(["_source_file", "Label"]).size().unstack(fill_value=0)
pivot.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")

ax.set_title("Distribución de Labels por Archivo")
ax.set_xlabel("Archivo")
ax.set_ylabel("Cantidad de flujos")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Estadísticas descriptivas de features clave

Features más relevantes para la Neural ODE (Sección 8 de la consigna).

In [ ]:
# Features clave para la Neural ODE
key_features = [
    # Dinámica temporal
    "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max", "Flow IAT Min",
    "Fwd IAT Mean", "Bwd IAT Mean",
    "Flow Bytes/s", "Flow Packets/s",
    "Active Mean", "Idle Mean",
    # Estructura del tráfico
    "Fwd Packet Length Mean", "Bwd Packet Length Mean",
    "Packet Length Variance",
    "Total Fwd Packets", "Total Backward Packets",
    # Comportamiento TCP
    "SYN Flag Count", "FIN Flag Count", "RST Flag Count",
    "Init_Win_bytes_forward", "Init_Win_bytes_backward",
]

# Filtrar features que existen en el dataset
available_features = [f for f in key_features if f in df_all.columns]
missing_features = [f for f in key_features if f not in df_all.columns]

if missing_features:
    print(f"Features no encontradas: {missing_features}\n")

# Convertir a numérico
for col in available_features:
    df_all[col] = pd.to_numeric(df_all[col], errors="coerce")

df_all[available_features].describe().T.round(2)

## 7. Distribución temporal de flujos

In [ ]:
# Parsear timestamps
if "Timestamp" in df_all.columns:
    df_all["Timestamp"] = pd.to_datetime(df_all["Timestamp"], errors="coerce", dayfirst=True)
    
    print(f"Rango temporal: {df_all['Timestamp'].min()} → {df_all['Timestamp'].max()}")
    print(f"Timestamps inválidos (NaT): {df_all['Timestamp'].isna().sum():,}")
    
    # Flujos por hora
    df_valid_ts = df_all.dropna(subset=["Timestamp"])
    flows_per_hour = df_valid_ts.set_index("Timestamp").resample("1h").size()
    
    fig, ax = plt.subplots(figsize=(16, 5))
    flows_per_hour.plot(ax=ax)
    ax.set_title("Flujos de red por hora")
    ax.set_ylabel("Cantidad de flujos")
    ax.set_xlabel("Tiempo")
    plt.tight_layout()
    plt.show()
else:
    print("Columna 'Timestamp' no encontrada.")

## 8. Visualizaciones: Benigno vs. DDoS

Comparación de distribuciones de features clave entre tráfico benigno y ataques DDoS/DoS.

In [ ]:
# Crear columna binaria: benigno vs. ataque
df_all["is_attack"] = (df_all["Label"] != "Benign").astype(int)

# Features a comparar
compare_features = [
    "Flow Packets/s", "Flow Bytes/s",
    "Flow IAT Mean", "Flow IAT Std",
    "Fwd Packet Length Mean", "Bwd Packet Length Mean",
    "Init_Win_bytes_forward", "Packet Length Variance",
]
compare_features = [f for f in compare_features if f in df_all.columns]

n_feats = len(compare_features)
fig, axes = plt.subplots(2, (n_feats + 1) // 2, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(compare_features):
    ax = axes[i]
    benign_vals = df_all.loc[df_all["is_attack"] == 0, feat].dropna()
    attack_vals = df_all.loc[df_all["is_attack"] == 1, feat].dropna()
    
    # Usar log scale si los valores tienen rango muy amplio
    use_log = benign_vals.max() > 1000 * (benign_vals.median() + 1)
    
    if use_log:
        benign_plot = np.log1p(benign_vals)
        attack_plot = np.log1p(attack_vals)
        ax.set_xlabel(f"log(1 + {feat})")
    else:
        benign_plot = benign_vals
        attack_plot = attack_vals
        ax.set_xlabel(feat)
    
    # Clip outliers extremos para visualización
    p99 = np.percentile(pd.concat([benign_plot, attack_plot]).dropna(), 99)
    bins = np.linspace(0, p99, 50)
    
    ax.hist(benign_plot.clip(upper=p99), bins=bins, alpha=0.5, label="Benign", density=True)
    ax.hist(attack_plot.clip(upper=p99), bins=bins, alpha=0.5, label="Attack", density=True)
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=8)

# Ocultar ejes sobrantes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribución de Features: Benigno vs. Ataque", fontsize=14)
plt.tight_layout()
plt.show()

## 9. Correlación entre features clave

In [ ]:
# Matriz de correlación de features clave
corr_features = [f for f in available_features if f in df_all.columns]
corr_matrix = df_all[corr_features].corr()

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr_matrix.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(corr_features)))
ax.set_yticks(range(len(corr_features)))
ax.set_xticklabels(corr_features, rotation=90, fontsize=8)
ax.set_yticklabels(corr_features, fontsize=8)
plt.colorbar(im, ax=ax, label="Correlación")
ax.set_title("Correlación entre Features Clave")
plt.tight_layout()
plt.show()

## 10. Resumen de NaN por columna

In [ ]:
# NaN totales por columna (solo mostrar las que tienen NaN)
nan_counts = df_all.isna().sum()
nan_nonzero = nan_counts[nan_counts > 0].sort_values(ascending=False)

if len(nan_nonzero) > 0:
    print("Columnas con valores faltantes:\n")
    for col, count in nan_nonzero.items():
        pct = count / len(df_all) * 100
        print(f"  {col:40s} {count:>10,} ({pct:.2f}%)")
else:
    print("No hay valores faltantes en el dataset.")

print(f"\nTotal de columnas: {len(df_all.columns)}")
print(f"Columnas numéricas: {len(df_all.select_dtypes(include=[np.number]).columns)}")